# Chronos-2 uncertainty-regularised LoRA fine-tuning

Ordinary LoRA fine-tuning asks the model only to fit the official quantile loss.
This experiment adds a second term that pushes Chronos-2 towards substantially
wider prediction intervals, to test whether forcing wider uncertainty reduces
downstream M6 overconfidence and improves RPS.

This is deliberately uncertainty regularisation, not calibration correction. The
coverage targets are set ABOVE the nominal levels of the intervals themselves:
the Q0.10 to Q0.90 interval is pushed towards 95 per cent coverage rather than
its nominal 80, and Q0.20 to Q0.80 towards 75 per cent rather than its nominal
60. A model that met those targets would be deliberately over-wide by
construction.

| | |
|---|---|
| Base model | `amazon/chronos-2`, reloaded fresh for every beta |
| Adapter | LoRA rank 8, alpha 16, official default target modules |
| Loss | official quantile loss + n_quantiles x beta x uncertainty penalty |
| Betas | 0.5, 1.0, 2.0, trained sequentially and independently |
| Data | Dataset E, the same panel and split as the ordinary LoRA run |

Everything else matches `chronos2_lora_training.ipynb` exactly: the same Dataset
E preprocessing, the same chronological split, the same LoRA settings, learning
rate, step count, seed, context length and prediction length. The only
difference between this experiment and that one is the added penalty term.

Run `chronos2_uncertainty_lora_m6.ipynb` afterwards.

## 1. Install Chronos-2 and PEFT

In [ ]:
%pip install -q "chronos-forecasting==2.3.1" "transformers>=4.41,<5" "peft>=0.11"

In [ ]:
%pip uninstall -y torchao

## 2. Imports and configuration

The base settings are identical to the ordinary LoRA run. The new block is the
uncertainty penalty: its two coverage targets, the smoothing width and the beta
values.

In [ ]:
import gc
import inspect
import json
import random
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from chronos import Chronos2Pipeline

# --- Base model, unchanged ---------------------------------------------------
MODEL_ID = "amazon/chronos-2"

# --- Fixed experiment settings, identical to the ordinary LoRA run -----------
CONTEXT_LENGTH = 512
MIN_PAST = 512
PREDICTION_LENGTH = 20
RANDOM_SEED = 42

LORA_RANK = 8
LORA_ALPHA = 16
LORA_TARGET_MODULES = None    # None = the official Chronos-2 defaults
LEARNING_RATE = 1e-5
MAX_STEPS = 1000
EVAL_EVERY = 100

# --- The uncertainty penalty -------------------------------------------------
# Target coverage is set deliberately ABOVE each interval's nominal level, so
# meeting it means being over-wide on purpose.
COVERAGE_TARGETS = {
    (0.10, 0.90): 0.95,       # nominal 80 per cent, pushed to 95
    (0.20, 0.80): 0.75,       # nominal 60 per cent, pushed to 75
}
# Smoothing width of the soft coverage indicator, in Chronos-2's internally
# normalised target scale. The penalty is computed inside the official loss, so
# predictions and targets are already on that scale when it is applied.
COVERAGE_SMOOTHING = 0.1
BETAS = [0.5, 1.0, 2.0]

OFFICIAL_ASSET_ORDER = [
    "ABBV", "ACN", "AEP", "AIZ", "ALLE", "AMAT", "AMP", "AMZN", "AVB", "AVY",
    "AXP", "BDX", "BF-B", "BMY", "BR", "CARR", "CDW", "CE", "CHTR", "CNC",
    "CNP", "COP", "CTAS", "CZR", "DG", "DPZ", "DRE", "DXC", "EWA", "EWC",
    "EWG", "EWH", "EWJ", "EWL", "EWQ", "EWT", "EWU", "EWY", "EWZ", "FTV",
    "GOOG", "GPC", "GSG", "HIG", "HIGH.L", "HST", "HYG", "IAU", "ICLN",
    "IEAA.L", "IEF", "IEFM.L", "IEMG", "IEUS", "IEVL.L", "IGF", "INDA",
    "IUMO.L", "IUVL.L", "IVV", "IWM", "IXN", "JPEA.L", "JPM", "KR", "LQD",
    "MCHI", "META", "MVEU.L", "OGN", "PG", "PPL", "PRU", "PYPL", "RE",
    "REET", "ROL", "ROST", "SEGA.L", "SHY", "SLV", "SPMV.L", "TLT", "UNH",
    "URI", "V", "VRSK", "VXX", "WRK", "XLB", "XLC", "XLE", "XLF", "XLI",
    "XLK", "XLP", "XLU", "XLV", "XLY", "XOM",
]
N_ASSETS = len(OFFICIAL_ASSET_ORDER)

# --- Chronological split, identical to the ordinary LoRA run -----------------
TRAIN_TARGET_CEILING = "2022-02-04"
VALIDATION_ORIGIN = "2022-02-04"
VALIDATION_TARGET_START = "2022-02-07"
VALIDATION_TARGET_END = "2022-03-04"
FIRST_M6_TARGET = "2022-03-07"

def reset_seeds(seed: int = RANDOM_SEED) -> list:
    """Reset every random source before a run, and report which were reset.

    Called immediately before each beta loads its base model, so the three
    configurations get the same initialisation and the same stochastic training
    sequence as far as the libraries allow. Strict deterministic algorithms are
    deliberately not enabled.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    reset = ["random", "numpy", "torch", "torch.cuda"]
    try:
        import transformers
        transformers.set_seed(seed)
        reset.append("transformers")
    except Exception:
        pass
    return reset


SEED_SOURCES = reset_seeds()

print("Versions")
print(f"  chronos-forecasting : {version('chronos-forecasting')}")
print(f"  transformers        : {version('transformers')}")
print(f"  torch               : {torch.__version__}")
print(f"\nBase model : {MODEL_ID}")
print(f"LoRA       : rank {LORA_RANK}, alpha {LORA_ALPHA}, lr {LEARNING_RATE}, "
      f"max_steps {MAX_STEPS}, eval every {EVAL_EVERY}, seed {RANDOM_SEED}")
print("Uncertainty penalty targets:")
for (lo, hi), target in COVERAGE_TARGETS.items():
    print(f"   Q{lo:.2f} to Q{hi:.2f}  nominal {hi - lo:.0%}  ->  target {target:.0%}")
print(f"Smoothing  : {COVERAGE_SMOOTHING} (normalised target scale)")
print(f"Betas      : {BETAS} (trained sequentially, each from a fresh base model)")
print(f"Seeded     : {SEED_SOURCES} at {RANDOM_SEED}, reset before every beta")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Paths

Dataset E is the same file the ordinary LoRA run used. Each beta gets its own
checkpoint folder, so no configuration can overwrite another or the completed
ordinary-LoRA checkpoint.

In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive/HonoursResearch")

# Same Dataset E file as the ordinary LoRA run
DATASET_E_PATH = Path("/content/drive/MyDrive/HonoursResearch/datasets/extended_eodhd_ohlcv_2000_2023.csv")

# One checkpoint folder per beta, separate from the ordinary LoRA checkpoint
CHECKPOINT_ROOT = Path("/content/drive/MyDrive/HonoursResearch/checkpoints/chronos2_uncertainty_lora")


def checkpoint_dir(beta: float) -> Path:
    return CHECKPOINT_ROOT / f"beta{beta}"


# Written later by chronos2_uncertainty_lora_m6.ipynb
M6_OUTPUT_ROOT = Path("/content/drive/MyDrive/HonoursResearch/outputs/chronos2_uncertainty_lora")

if not DATASET_E_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset E not found at:\n  {DATASET_E_PATH}\n\n"
        "This is the same file the ordinary LoRA run used. In the repository it "
        f"is Quasi-Average Reimplementation DRE-as-Stock/data/\n"
        "adjusted_close_panel.csv"
    )

print(f"Dataset E (input)      : {DATASET_E_PATH}")
print(f"Checkpoint root (new)  : {CHECKPOINT_ROOT}")
for beta in BETAS:
    checkpoint_dir(beta).mkdir(parents=True, exist_ok=True)
    print(f"   beta {beta} -> {checkpoint_dir(beta)}")
print(f"M6 outputs (later)     : {M6_OUTPUT_ROOT}")
print(f"\nDataset E found: {DATASET_E_PATH.stat().st_size / 1e6:.1f} MB")

## 5. GPU check

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Runtime > Change runtime type > Hardware "
        "accelerator: GPU (an L4 is recommended), then re-run from section 3."
    )

GPU_NAME = torch.cuda.get_device_name(0)
print(f"GPU    : {GPU_NAME}")
print(f"memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 6. Require PEFT and LoRA support

Identical guard to the ordinary LoRA run. Training must not fall back to full
fine-tuning.

In [ ]:
try:
    import peft
except ImportError as exc:
    raise RuntimeError(
        "PEFT is not available, so LoRA fine-tuning cannot run. Install it with "
        "`%pip install peft>=0.11` and re-run. Training must NOT proceed without "
        "it: the run would silently become full-model fine-tuning."
    ) from exc

print(f"peft {peft.__version__}")

FIT_SIGNATURE = inspect.signature(Chronos2Pipeline.fit)
FIT_PARAMS = list(FIT_SIGNATURE.parameters)
print(f"\nChronos2Pipeline.fit parameters:\n  {FIT_PARAMS}")

LORA_HINTS = [p for p in FIT_PARAMS
              if any(k in p.lower() for k in ("lora", "peft", "adapter"))]
if not LORA_HINTS:
    raise RuntimeError(
        "Chronos2Pipeline.fit exposes no LoRA/PEFT parameter in this build:\n"
        f"  {FIT_PARAMS}\n"
        "Calling fit() now would fine-tune the FULL model. Stop and confirm the "
        "correct API before training."
    )
print(f"LoRA-related fit parameters found: {LORA_HINTS}")

## 7. Load Dataset E and build daily log returns

Identical to the ordinary LoRA run: forward-fill only after each asset's genuine
inception, then `log(P[t] / P[t-1])`. Nothing is backfilled, interpolated,
clipped, winsorised or rescaled, and the genuine leading NaNs are preserved.

In [ ]:
panel = pd.read_csv(DATASET_E_PATH, parse_dates=["date"]).set_index("date")

assert list(panel.columns) == OFFICIAL_ASSET_ORDER, (
    "Dataset E columns are not the 100 official M6 assets in official order"
)
assert panel.index.is_monotonic_increasing, "Dataset E dates are not ascending"
assert not panel.index.duplicated().any(), "Dataset E has duplicate dates"
assert set(panel.index.dayofweek) <= {0, 1, 2, 3, 4}, "Dataset E is not a weekday panel"

filled = panel.ffill()
returns = np.log(filled / filled.shift(1))
returns = returns.where(filled.notna() & filled.shift(1).notna())

leading_total, interior_total = 0, 0
for symbol in returns.columns:
    values = returns[symbol].to_numpy()
    n_lead = int(np.argmax(~np.isnan(values))) if np.isnan(values[0]) else 0
    leading_total += n_lead
    interior_total += int(np.isnan(values[n_lead:]).sum())
assert interior_total == 0, (
    f"Dataset E produced {interior_total} interior NaNs; the panel is not the "
    "expected forward-filled weekday panel"
)

DATE_INDEX = returns.index
print(f"Dataset E: {panel.shape[0]} weekday rows x {panel.shape[1]} assets")
print(f"  {DATE_INDEX.min().date()} .. {DATE_INDEX.max().date()}")
print(f"  leading NaN (genuine pre-inception) : {leading_total:,}")
print(f"  interior NaN                        : {interior_total}")

## 8. Chronological split

The same split as the ordinary LoRA run, derived from Dataset E's own index and
checked rather than assumed.

In [ ]:
def position_of(date_string: str) -> int:
    stamp = pd.Timestamp(date_string)
    if stamp not in DATE_INDEX:
        raise ValueError(f"{date_string} is not a row in Dataset E's calendar")
    return int(DATE_INDEX.get_loc(stamp))


train_end_pos = position_of(TRAIN_TARGET_CEILING)
validation_origin_pos = position_of(VALIDATION_ORIGIN)

validation_target = DATE_INDEX[validation_origin_pos + 1:
                               validation_origin_pos + 1 + PREDICTION_LENGTH]
assert len(validation_target) == PREDICTION_LENGTH
assert validation_target[0] == pd.Timestamp(VALIDATION_TARGET_START)
assert validation_target[-1] == pd.Timestamp(VALIDATION_TARGET_END)
assert validation_target[-1] < pd.Timestamp(FIRST_M6_TARGET), (
    "the validation target overlaps the first M6 target"
)

validation_end_pos = validation_origin_pos + PREDICTION_LENGTH

# One multivariate task each: (1, 100, time), assets on the variate axis.
train_array = returns.to_numpy(dtype=np.float32).T[np.newaxis, :, :train_end_pos + 1]
validation_array = returns.to_numpy(dtype=np.float32).T[np.newaxis, :, :validation_end_pos + 1]

assert train_array.shape == (1, N_ASSETS, train_end_pos + 1)
assert validation_array.shape == (1, N_ASSETS, validation_end_pos + 1)

# Kept for the post-training coverage report in section 12.
VALIDATION_CONTEXT = returns.to_numpy(dtype=np.float32).T[
    np.newaxis, :, validation_origin_pos + 1 - CONTEXT_LENGTH: validation_origin_pos + 1]
VALIDATION_ACTUAL = returns.to_numpy(dtype=np.float64).T[
    :, validation_origin_pos + 1: validation_end_pos + 1]
assert VALIDATION_CONTEXT.shape == (1, N_ASSETS, CONTEXT_LENGTH)
assert VALIDATION_ACTUAL.shape == (N_ASSETS, PREDICTION_LENGTH)

print(f"training array   : {train_array.shape}  "
      f"{DATE_INDEX[0].date()} .. {DATE_INDEX[train_end_pos].date()}")
print(f"validation array : {validation_array.shape}  "
      f"{DATE_INDEX[0].date()} .. {DATE_INDEX[validation_end_pos].date()}")
print(f"  training targets never pass : {TRAIN_TARGET_CEILING}")
print(f"  validation target           : {validation_target[0].date()} .. "
      f"{validation_target[-1].date()}")
print("\nBoundary checks passed.")

## 9. Map the configuration onto the installed `fit()`

Identical mapping to the ordinary LoRA run. Internal validation is required, so
the run stops if the installed build exposes no validation-inputs parameter.

In [ ]:
import inspect

# Explicit Chronos-2 LoRA target modules.
# This overrides the earlier None value.
LORA_TARGET_MODULES = [
    "self_attention.q",
    "self_attention.v",
    "self_attention.k",
    "self_attention.o",
    "output_patch_embedding.output_layer",
]

REQUIRED_FIT_PARAMS = [
    "inputs",
    "prediction_length",
    "validation_inputs",
    "finetune_mode",
    "lora_config",
    "context_length",
    "learning_rate",
    "num_steps",
    "output_dir",
    "min_past",
]

missing_fit_params = [
    name for name in REQUIRED_FIT_PARAMS
    if name not in FIT_PARAMS
]

if missing_fit_params:
    raise RuntimeError(
        "The installed Chronos-2 fit() API is missing required parameters:\n  "
        + "\n  ".join(missing_fit_params)
        + f"\n\nActual signature:\n  {FIT_SIGNATURE}"
    )

# fit() accepts additional Hugging Face Trainer settings through
# **extra_trainer_kwargs.
HAS_EXTRA_TRAINER_KWARGS = any(
    parameter.kind == inspect.Parameter.VAR_KEYWORD
    for parameter in FIT_SIGNATURE.parameters.values()
)

LORA_CONFIG = {
    "r": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "target_modules": LORA_TARGET_MODULES,
}

BASE_FIT_KWARGS = {
    "finetune_mode": "lora",
    "lora_config": LORA_CONFIG,
    "learning_rate": LEARNING_RATE,
    "num_steps": MAX_STEPS,
    "context_length": CONTEXT_LENGTH,
    "min_past": MIN_PAST,
    "prediction_length": PREDICTION_LENGTH,
}

if "eval_steps" in FIT_PARAMS or HAS_EXTRA_TRAINER_KWARGS:
    BASE_FIT_KWARGS["eval_steps"] = EVAL_EVERY
else:
    raise RuntimeError(
        "fit() cannot accept eval_steps, so the validation interval "
        "cannot be configured safely."
    )

if "seed" in FIT_PARAMS or HAS_EXTRA_TRAINER_KWARGS:
    BASE_FIT_KWARGS["seed"] = RANDOM_SEED
else:
    raise RuntimeError(
        "fit() cannot accept the requested Trainer seed."
    )

DATA_PARAM = "inputs"
VALIDATION_PARAM = "validation_inputs"

print("Resolved fit() arguments:")
print(f"  {'LoRA mode':<20} -> {'finetune_mode':<30} = lora")
print(f"  {'LoRA rank':<20} -> {'lora_config.r':<30} = {LORA_RANK}")
print(
    f"  {'LoRA alpha':<20} -> "
    f"{'lora_config.lora_alpha':<30} = {LORA_ALPHA}"
)
print(
    f"  {'LoRA target modules':<20} -> "
    f"{'lora_config.target_modules':<30} = {LORA_TARGET_MODULES}"
)
print(
    f"  {'learning rate':<20} -> "
    f"{'learning_rate':<30} = {LEARNING_RATE}"
)
print(
    f"  {'max steps':<20} -> "
    f"{'num_steps':<30} = {MAX_STEPS}"
)
print(
    f"  {'eval interval':<20} -> "
    f"{'eval_steps':<30} = {EVAL_EVERY}"
)
print(
    f"  {'seed':<20} -> "
    f"{'seed':<30} = {RANDOM_SEED}"
)
print(
    f"  {'context length':<20} -> "
    f"{'context_length':<30} = {CONTEXT_LENGTH}"
)
print(
    f"  {'min past':<20} -> "
    f"{'min_past':<30} = {MIN_PAST}"
)
print(
    f"  {'prediction length':<20} -> "
    f"{'prediction_length':<30} = {PREDICTION_LENGTH}"
)

print(f"\ntraining data parameter   : {DATA_PARAM}")
print(f"validation data parameter : {VALIDATION_PARAM}")
print("\nLoRA and internal validation configuration passed.")

## 10. Confirm the quantile levels

The penalty needs the 0.10, 0.20, 0.80 and 0.90 columns. Their positions are
read from the loaded model and verified before any training starts, never
assumed. The model is released again immediately: every beta run loads its own
fresh copy.

In [ ]:
probe = Chronos2Pipeline.from_pretrained(MODEL_ID, device_map="cpu")
QUANTILE_LEVELS = np.asarray(probe.quantiles, dtype=np.float64)
N_QUANTILES = int(QUANTILE_LEVELS.size)

# The four levels the penalty needs, resolved from the model's own tensor and
# then asserted to be exactly those values rather than merely close neighbours.
REQUIRED_LEVELS = (0.10, 0.20, 0.80, 0.90)
QUANTILE_INDEX = {}
for level in REQUIRED_LEVELS:
    hits = np.flatnonzero(np.isclose(QUANTILE_LEVELS, level, atol=1e-9))
    if hits.size != 1:
        raise RuntimeError(
            f"Quantile level {level} is not uniquely present in this model's "
            f"levels: {QUANTILE_LEVELS.tolist()}. The uncertainty penalty "
            "cannot be formed."
        )
    QUANTILE_INDEX[level] = int(hits[0])
    resolved_level = float(QUANTILE_LEVELS[QUANTILE_INDEX[level]])
    assert abs(resolved_level - level) < 1e-9, (
        f"index {QUANTILE_INDEX[level]} holds level {resolved_level}, not {level}"
    )

del probe
gc.collect()

print(f"quantile levels ({N_QUANTILES}): {QUANTILE_LEVELS.tolist()}")
print("penalty columns, index -> level actually stored there:")
for level in REQUIRED_LEVELS:
    print(f"   Q{level:.2f} -> index {QUANTILE_INDEX[level]} "
          f"= {QUANTILE_LEVELS[QUANTILE_INDEX[level]]}")
print(f"loss scale factor (number of quantiles) = {N_QUANTILES}")

## 11. The uncertainty-regularised loss

The hook targets one exact method, `Chronos2Model._compute_loss`, and stops if
it is absent or its signature does not carry the six arguments the penalty needs.
Nothing is selected by pattern matching.

The official loss is called unchanged and the penalty is added to its result:

    total = official_compute_loss + n_quantiles * beta * penalty

**Putting the target on the same scale as the predictions.** `_compute_loss`
receives `quantile_preds` already normalised, and normalises the target itself
with `self.instance_norm(future_target, loc_scale)`. That call does not mutate
the caller's `future_target`, so a wrapper that reads the raw argument would be
comparing normalised predictions against unnormalised targets. The penalty
therefore applies the identical `self.instance_norm(future_target, loc_scale)`
to its own copy before measuring coverage, reusing the model's own normaliser
and the same `loc_scale` rather than approximating it.

The effective loss mask is rebuilt the same way too: the future-target mask
intersected with the inverse of the patched future-covariates mask, so positions
that are missing targets, and positions that are known future covariates rather
than things to predict, are both excluded. Target and mask are unsqueezed to the
official `batch x 1 x horizon` and padded to the prediction horizon with masked
out positions, exactly as padding for loss does.

Shapes are asserted, not discovered:

    quantile predictions : batch x quantiles x horizon
    normalised target    : batch x 1 x horizon
    effective mask       : batch x 1 x horizon

For each of the 20 forecast-horizon positions, soft coverage is computed across
that horizon's valid observations, for both intervals:

    cov = sum over valid i of
              sigmoid((y_i - q_lo_i) / s) * sigmoid((q_hi_i - y_i) / s)
          / sum over valid i of 1

with `s = 0.1` on the normalised scale. A smooth indicator is needed because a
hard in-or-out count has zero gradient everywhere.

The penalty averages the two intervals at each horizon, then averages over
horizons:

    shortfall = relu(target_coverage - cov)
    penalty   = mean over horizons of
                    (shortfall_95^2 + shortfall_75^2) / 2

Dividing by two keeps beta on its intended scale: without it the two intervals
would sum and every beta would act twice as strongly as its label suggests. A
horizon that already exceeds both targets contributes nothing, so the term can
only widen intervals, never narrow them.

The wrapper counts its calls and prints one diagnostic line on the first call of
each beta run, so the penalty can be seen to be live and finite without flooding
the training log.

In [ ]:
import chronos.chronos2.model as c2model

# --- target the exact official method, never a pattern match ----------------
MODEL_CLASS = getattr(c2model, "Chronos2Model", None)
if MODEL_CLASS is None:
    raise RuntimeError(
        "chronos.chronos2.model has no Chronos2Model class in this build. The "
        f"uncertainty penalty cannot be attached. Found: "
        f"{[n for n in vars(c2model) if isinstance(vars(c2model)[n], type)]}"
    )

LOSS_METHOD_NAME = "_compute_loss"
ORIGINAL_COMPUTE_LOSS = getattr(MODEL_CLASS, LOSS_METHOD_NAME, None)
if ORIGINAL_COMPUTE_LOSS is None:
    raise RuntimeError(
        f"Chronos2Model has no {LOSS_METHOD_NAME} in this build. Loss-related "
        f"attributes present: "
        f"{[a for a in dir(MODEL_CLASS) if 'loss' in a.lower()]}"
    )

LOSS_SIGNATURE = inspect.signature(ORIGINAL_COMPUTE_LOSS)
REQUIRED_LOSS_ARGS = [
    "quantile_preds",
    "future_target",
    "future_target_mask",
    "patched_future_covariates_mask",
    "loc_scale",
    "num_output_patches",
]

missing_args = [
    argument
    for argument in REQUIRED_LOSS_ARGS
    if argument not in LOSS_SIGNATURE.parameters
]

if missing_args:
    raise RuntimeError(
        f"Chronos2Model.{LOSS_METHOD_NAME} is missing the arguments the penalty "
        f"needs: {missing_args}\n"
        f"  actual signature: {LOSS_SIGNATURE}\n"
        "The wrapper cannot be attached safely. Stop and reconcile the API."
    )

print(
    f"hooking {MODEL_CLASS.__name__}."
    f"{LOSS_METHOD_NAME}{LOSS_SIGNATURE}"
)
print(f"all required arguments present: {REQUIRED_LOSS_ARGS}\n")
print(inspect.getsource(ORIGINAL_COMPUTE_LOSS))


def _align_horizon(tensor, horizon: int, fill):
    """Pad or trim the last axis to the prediction horizon.

    Padding contributes nothing to the custom loss because padded mask
    positions are false.
    """
    current = tensor.shape[-1]

    if current == horizon:
        return tensor

    if current > horizon:
        return tensor[..., :horizon]

    pad = tensor.new_full(
        tensor.shape[:-1] + (horizon - current,),
        fill,
    )
    return torch.cat([tensor, pad], dim=-1)


def uncertainty_penalty(
    model,
    quantile_preds,
    future_target,
    future_target_mask,
    patched_future_covariates_mask,
    loc_scale,
):
    """Average squared coverage shortfall across valid horizon positions."""

    batch, n_quantiles, horizon = quantile_preds.shape

    assert n_quantiles == N_QUANTILES, (
        f"quantile_preds has {n_quantiles} quantiles, "
        f"expected {N_QUANTILES}"
    )

    # Use the model's official target normalisation.
    normed_target, _ = model.instance_norm(
        future_target,
        loc_scale,
    )

    # Calculate the custom penalty in float32 while preserving gradients.
    quantile_preds = quantile_preds.float()
    normed_target = normed_target.to(
        device=quantile_preds.device,
        dtype=torch.float32,
    )

    # Official target layout: batch x 1 x target horizon.
    if normed_target.dim() == 2:
        normed_target = normed_target.unsqueeze(1)

    if normed_target.dim() != 3 or normed_target.shape[1] != 1:
        raise RuntimeError(
            "Unexpected normalised-target shape: "
            f"{tuple(normed_target.shape)}. "
            "Expected batch x 1 x horizon."
        )

    # Match the official Chronos-2 loss:
    # when no mask is supplied, valid targets are the non-NaN targets.
    if future_target_mask is None:
        target_mask = ~torch.isnan(normed_target)
    else:
        target_mask = future_target_mask.to(
            device=quantile_preds.device,
            dtype=torch.bool,
        )

        if target_mask.dim() == 2:
            target_mask = target_mask.unsqueeze(1)

    if target_mask.dim() != 3 or target_mask.shape[1] != 1:
        raise RuntimeError(
            "Unexpected future-target-mask shape: "
            f"{tuple(target_mask.shape)}. "
            "Expected batch x 1 x horizon."
        )

    # Prevent masked NaNs from entering sigmoid calculations.
    normed_target = torch.where(
        target_mask,
        normed_target,
        torch.zeros_like(normed_target),
    )

    # The model pads a 20-day target to its 32-position patch horizon.
    normed_target = _align_horizon(
        normed_target,
        horizon,
        0.0,
    )
    target_mask = _align_horizon(
        target_mask,
        horizon,
        False,
    )

    # The official covariate mask arrives as:
    # batch x number_of_patches x output_patch_size.
    # Flatten the final two dimensions to match the loss horizon.
    if patched_future_covariates_mask is None:
        covariate_mask = torch.zeros_like(
            target_mask,
            dtype=torch.bool,
        )
    else:
        covariate_mask = patched_future_covariates_mask.to(
            device=quantile_preds.device,
            dtype=torch.bool,
        )

        if covariate_mask.dim() == 3:
            if covariate_mask.shape[0] != batch:
                raise RuntimeError(
                    "Covariate-mask batch does not match predictions: "
                    f"{tuple(covariate_mask.shape)} versus batch {batch}."
                )

            covariate_mask = covariate_mask.reshape(
                batch,
                1,
                -1,
            )

        elif covariate_mask.dim() == 2:
            covariate_mask = covariate_mask.unsqueeze(1)

        else:
            raise RuntimeError(
                "Unexpected patched-future-covariate-mask shape: "
                f"{tuple(covariate_mask.shape)}."
            )

        covariate_mask = _align_horizon(
            covariate_mask,
            horizon,
            False,
        )

    effective_mask = target_mask & (~covariate_mask)

    assert normed_target.shape == (batch, 1, horizon), (
        f"normalised target {tuple(normed_target.shape)}, "
        f"expected {(batch, 1, horizon)}"
    )

    assert target_mask.shape == (batch, 1, horizon), (
        f"target mask {tuple(target_mask.shape)}, "
        f"expected {(batch, 1, horizon)}"
    )

    assert covariate_mask.shape == (batch, 1, horizon), (
        f"covariate mask {tuple(covariate_mask.shape)}, "
        f"expected {(batch, 1, horizon)}"
    )

    assert effective_mask.shape == (batch, 1, horizon), (
        f"effective mask {tuple(effective_mask.shape)}, "
        f"expected {(batch, 1, horizon)}"
    )

    y = normed_target[:, 0, :]
    weight = effective_mask[:, 0, :].to(torch.float32)
    per_horizon_valid = weight.sum(dim=0)

    total = torch.zeros(
        (),
        dtype=torch.float32,
        device=quantile_preds.device,
    )

    counted = 0

    soft_sums = {
        interval: 0.0
        for interval in COVERAGE_TARGETS
    }

    hard_sums = {
        interval: 0.0
        for interval in COVERAGE_TARGETS
    }

    for horizon_index in range(horizon):
        denominator = per_horizon_valid[horizon_index]

        # The 20-day target is padded to 32 positions. The final 12 padded
        # positions have zero valid observations and are excluded.
        if float(denominator) <= 0.0:
            continue

        shortfall_sq = torch.zeros(
            (),
            dtype=torch.float32,
            device=quantile_preds.device,
        )

        for interval, target_coverage in COVERAGE_TARGETS.items():
            lower_level, upper_level = interval

            lower = quantile_preds[
                :,
                QUANTILE_INDEX[lower_level],
                horizon_index,
            ]

            upper = quantile_preds[
                :,
                QUANTILE_INDEX[upper_level],
                horizon_index,
            ]

            inside = (
                torch.sigmoid(
                    (y[:, horizon_index] - lower)
                    / COVERAGE_SMOOTHING
                )
                * torch.sigmoid(
                    (upper - y[:, horizon_index])
                    / COVERAGE_SMOOTHING
                )
            )

            coverage = (
                inside * weight[:, horizon_index]
            ).sum() / denominator

            shortfall = torch.clamp(
                target_coverage - coverage,
                min=0.0,
            )

            shortfall_sq = shortfall_sq + shortfall**2
            soft_sums[interval] += float(coverage.detach())

            with torch.no_grad():
                hard = (
                    (
                        (y[:, horizon_index] >= lower)
                        & (y[:, horizon_index] <= upper)
                    ).to(torch.float32)
                    * weight[:, horizon_index]
                ).sum() / denominator

            hard_sums[interval] += float(hard)

        # Average the two interval penalties.
        total = (
            total
            + shortfall_sq / len(COVERAGE_TARGETS)
        )
        counted += 1

    if counted == 0:
        raise RuntimeError(
            "The uncertainty penalty found zero valid forecast horizons."
        )

    detail = {
        "horizons_counted": counted,
        "valid_per_horizon_min": int(
            per_horizon_valid[
                per_horizon_valid > 0
            ].min().item()
        ),
        "valid_per_horizon_mean": float(
            per_horizon_valid[
                per_horizon_valid > 0
            ].mean().item()
        ),
        "valid_per_horizon_max": int(
            per_horizon_valid.max().item()
        ),
        "soft_coverage": {
            f"Q{lower:.2f}-Q{upper:.2f}":
                soft_sums[(lower, upper)] / counted
            for (lower, upper) in COVERAGE_TARGETS
        },
        "hard_coverage": {
            f"Q{lower:.2f}-Q{upper:.2f}":
                hard_sums[(lower, upper)] / counted
            for (lower, upper) in COVERAGE_TARGETS
        },
        "batch": int(batch),
        "n_quantiles": int(n_quantiles),
        "horizon": int(horizon),
        "pred_dtype": str(quantile_preds.dtype),
        "penalty_dtype": str(total.dtype),
    }

    return total / counted, detail

def make_uncertainty_loss(beta: float):
    """Official loss plus n_quantiles x beta x uncertainty penalty."""
    state = {
        "calls": 0,
        "reported": False,
        "first_call": None,
    }

    def wrapped(self, *args, **kwargs):
        base = ORIGINAL_COMPUTE_LOSS(
            self,
            *args,
            **kwargs,
        )

        state["calls"] += 1

        if beta == 0.0:
            return base

        bound = LOSS_SIGNATURE.bind(
            self,
            *args,
            **kwargs,
        )
        bound.apply_defaults()
        arguments = bound.arguments

        penalty, detail = uncertainty_penalty(
            self,
            arguments["quantile_preds"],
            arguments["future_target"],
            arguments["future_target_mask"],
            arguments["patched_future_covariates_mask"],
            arguments["loc_scale"],
        )

        scale_factor = N_QUANTILES * beta
        scaled = scale_factor * penalty

        # The official loss remains unchanged; the combined scalar is float32.
        total = base.float() + scaled

        # Print one diagnostic on the first real call of each beta.
        if not state["reported"]:
            state["reported"] = True

            predictions = arguments["quantile_preds"]

            diagnostic_target, _ = self.instance_norm(
                arguments["future_target"],
                arguments["loc_scale"],
            )

            if diagnostic_target.dim() == 2:
                normalised_target_shape = tuple(
                    diagnostic_target.unsqueeze(1).shape
                )
            else:
                normalised_target_shape = tuple(
                    diagnostic_target.shape
                )

            # future_target_mask may legitimately be None.
            diagnostic_mask = arguments["future_target_mask"]

            if diagnostic_mask is None:
                mask_shape = (
                    "None (implicit all-valid mask for target shape "
                    f"{tuple(arguments['future_target'].shape)})"
                )
            else:
                mask_shape = str(
                    tuple(diagnostic_mask.shape)
                )

            finite = all(
                torch.isfinite(component).all().item()
                for component in (
                    base,
                    penalty,
                    scaled,
                    total,
                )
            )

            print(f"\n  [beta {beta}] first real loss call")
            print(
                f"    shapes   preds {tuple(predictions.shape)}  "
                f"normalised target {normalised_target_shape}  "
                f"mask {mask_shape}"
            )
            print(
                f"    sizes    batch {detail['batch']}  "
                f"quantiles {detail['n_quantiles']}  "
                f"horizon {detail['horizon']}"
            )
            print(
                f"    dtypes   preds {detail['pred_dtype']}  "
                f"penalty {detail['penalty_dtype']}  "
                f"base {base.dtype}"
            )
            print(
                "    valid obs per horizon  "
                f"min {detail['valid_per_horizon_min']}  "
                f"mean {detail['valid_per_horizon_mean']:.1f}  "
                f"max {detail['valid_per_horizon_max']}  "
                f"(horizons counted "
                f"{detail['horizons_counted']})"
            )

            for (
                lower_level,
                upper_level,
            ), target_coverage in COVERAGE_TARGETS.items():
                key = (
                    f"Q{lower_level:.2f}-"
                    f"Q{upper_level:.2f}"
                )

                print(
                    f"    {key}  "
                    f"soft {detail['soft_coverage'][key]:.4f}  "
                    f"hard {detail['hard_coverage'][key]:.4f}  "
                    f"target {target_coverage:.2f}"
                )

            print(
                f"    base official loss   : "
                f"{float(base):.6f}"
            )
            print(
                f"    unscaled penalty     : "
                f"{float(penalty):.6f}"
            )
            print(
                f"    scale factor         : {scale_factor} "
                f"(= {N_QUANTILES} quantiles x beta {beta})"
            )
            print(
                f"    scaled contribution  : "
                f"{float(scaled):.6f}"
            )
            print(
                f"    total loss           : "
                f"{float(total):.6f}"
            )
            print(
                f"    all components finite: {finite}"
            )
            print(
                "    hard coverage is a detached diagnostic only; "
                "the penalty uses the smooth estimate"
            )

            if not finite:
                raise RuntimeError(
                    "A non-finite loss component occurred on the "
                    "first call. Stop and inspect before training."
                )

            state["first_call"] = {
                "beta": beta,
                "batch": detail["batch"],
                "n_quantiles": detail["n_quantiles"],
                "horizon": detail["horizon"],
                "pred_dtype": detail["pred_dtype"],
                "penalty_dtype": detail["penalty_dtype"],
                "base_dtype": str(base.dtype),
                "horizons_counted":
                    detail["horizons_counted"],
                "valid_per_horizon_min":
                    detail["valid_per_horizon_min"],
                "valid_per_horizon_mean": round(
                    detail["valid_per_horizon_mean"],
                    2,
                ),
                "valid_per_horizon_max":
                    detail["valid_per_horizon_max"],
                "soft_coverage": {
                    key: round(value, 6)
                    for key, value
                    in detail["soft_coverage"].items()
                },
                "hard_coverage": {
                    key: round(value, 6)
                    for key, value
                    in detail["hard_coverage"].items()
                },
                "base_loss": round(
                    float(base),
                    6,
                ),
                "unscaled_penalty": round(
                    float(penalty),
                    6,
                ),
                "scale_factor": scale_factor,
                "scaled_contribution": round(
                    float(scaled),
                    6,
                ),
                "total_loss": round(
                    float(total),
                    6,
                ),
                "all_finite": bool(finite),
            }

        return total

    wrapped.__name__ = LOSS_METHOD_NAME
    wrapped.__wrapped_original__ = ORIGINAL_COMPUTE_LOSS
    wrapped.__call_state__ = state

    return wrapped


print(
    "\nLoss wrapper ready. "
    "It is installed and removed around each beta run."
)

## 12. Train beta 0.5, 1.0 and 2.0 sequentially

Each beta reloads a fresh `amazon/chronos-2` and trains from it, so no
configuration ever continues from another's weights. The training data, seed and
all other settings are identical across the three runs: only beta changes.

A configuration is skipped only when its checkpoint and metadata both exist and
pass verification, so an interrupted session resumes without redoing finished
work. GPU memory is released between runs.

After each run the saved checkpoint is reloaded and used to forecast the
validation window, and the realised interval coverage and mean interval width
are reported for both intervals. Quantile crossings are counted and reported,
never silently corrected.

In [ ]:
def checkpoint_ready(beta: float) -> bool:
    """True if this beta is already trained, saved and verifiable."""
    directory = checkpoint_dir(beta)
    metadata_file = directory / "training_metadata.json"
    if not metadata_file.is_file():
        return False
    try:
        meta = json.loads(metadata_file.read_text(encoding="utf-8"))
    except Exception:
        return False
    if not any(p.suffix in (".safetensors", ".bin", ".pt") or p.is_dir()
               for p in directory.iterdir()):
        return False
    return (meta.get("beta") == beta
            and meta.get("base_model_id") == MODEL_ID
            and meta.get("random_seed") == RANDOM_SEED
            and meta.get("max_steps") == MAX_STEPS
            and meta.get("lora_rank") == LORA_RANK)


def validation_coverage(model) -> dict:
    """Realised interval coverage and width on the validation window."""
    outputs = model.predict(inputs=VALIDATION_CONTEXT,
                            prediction_length=PREDICTION_LENGTH,
                            cross_learning=False)
    raw = outputs[0]
    raw = raw.to(torch.float32).cpu().numpy() if torch.is_tensor(raw) else np.asarray(raw)
    if raw.shape == (N_ASSETS, N_QUANTILES, PREDICTION_LENGTH):
        q = np.transpose(raw, (0, 2, 1))
    elif raw.shape == (N_ASSETS, PREDICTION_LENGTH, N_QUANTILES):
        q = raw
    else:
        raise ValueError(f"unexpected validation output shape {raw.shape}")

    crossings = int((np.diff(q, axis=2) < -1e-6).sum())
    valid = np.isfinite(VALIDATION_ACTUAL)
    report = {"quantile_crossings": crossings,
              "valid_observations": int(valid.sum())}
    for (lo, hi), target_coverage in COVERAGE_TARGETS.items():
        low = q[:, :, QUANTILE_INDEX[lo]]
        high = q[:, :, QUANTILE_INDEX[hi]]
        covered = (VALIDATION_ACTUAL >= low) & (VALIDATION_ACTUAL <= high) & valid
        key = f"Q{lo:.2f}-Q{hi:.2f}"
        report[key] = {
            "nominal": hi - lo,
            "target": target_coverage,
            "empirical_coverage": float(covered.sum() / valid.sum()),
            "mean_width": float(np.mean((high - low)[valid])),
            "median_width": float(np.median((high - low)[valid])),
        }
    return report


RUN_SUMMARIES = {}

for beta in BETAS:
    directory = checkpoint_dir(beta)
    print(f"\n{'=' * 74}\nbeta = {beta}\n{'=' * 74}")

    if checkpoint_ready(beta):
        meta = json.loads((directory / "training_metadata.json").read_text(encoding="utf-8"))
        RUN_SUMMARIES[beta] = meta
        print("  already trained and verified, skipped")
        continue

    # Reset every random source BEFORE the model loads, so LoRA initialisation
    # and the stochastic training sequence start identically for each beta.
    seeded = reset_seeds(RANDOM_SEED)
    print(f"  seeds reset at {RANDOM_SEED}: {seeded}")

    # A fresh base model for every beta: never continue from another run.
    pipeline = Chronos2Pipeline.from_pretrained(MODEL_ID, device_map="cuda")
    print(f"  loaded a fresh {MODEL_ID}")

    # Install the wrapper for this beta only.
    wrapper = make_uncertainty_loss(beta)
    setattr(MODEL_CLASS, LOSS_METHOD_NAME, wrapper)
    print(f"  loss patched: official + {N_QUANTILES} x {beta} x penalty")

    try:
        started = datetime.now(timezone.utc)
        call_kwargs = dict(BASE_FIT_KWARGS)
        call_kwargs[VALIDATION_PARAM] = validation_array
        fit_result = pipeline.fit(train_array, **call_kwargs)
        tuned = fit_result if isinstance(fit_result, Chronos2Pipeline) else pipeline
        elapsed = (datetime.now(timezone.utc) - started).total_seconds() / 60
        print(f"  trained in {elapsed:.1f} min")
    finally:
        # Always restore, even if training raised, so the next beta is clean.
        setattr(MODEL_CLASS, LOSS_METHOD_NAME, ORIGINAL_COMPUTE_LOSS)
        print("  official loss restored")

    # The wrapper must actually have been used, or this run is just
    # ordinary LoRA wearing the wrong label.
    wrapper_calls = wrapper.__call_state__["calls"]
    assert wrapper_calls > 0, (
        f"beta {beta}: the uncertainty wrapper was never called, so the "
        "penalty had no effect. Do not use this checkpoint."
    )
    print(f"  wrapper invoked {wrapper_calls:,} times")

    # --- LoRA safety check ---------------------------------------------------
    total = sum(p.numel() for p in tuned.model.parameters())
    trainable_params = [(n, p.numel()) for n, p in tuned.model.named_parameters()
                        if p.requires_grad]
    trainable = sum(n for _, n in trainable_params)
    share = trainable / total if total else 0.0
    adapter_like = [n for n, _ in trainable_params
                    if any(k in n.lower() for k in ("lora", "adapter"))]
    print(f"  trainable {trainable:,}/{total:,} ({share:.2%}), "
          f"adapter tensors {len(adapter_like)}")
    assert trainable > 0, "fit() adapted nothing"
    assert share < 0.25, (
        f"{share:.1%} of parameters are trainable: this looks like full "
        "fine-tuning, not LoRA. Do not use this checkpoint."
    )
    assert adapter_like, "no trainable parameter is named like a LoRA adapter"

    # --- save ----------------------------------------------------------------
    saver = next((getattr(tuned, m) for m in
                  ("save_pretrained", "save_adapter", "save")
                  if hasattr(tuned, m)), None)
    if saver is None:
        raise RuntimeError("the tuned pipeline exposes no save method")
    saver(str(directory))

    metadata = {
        "experiment": "uncertainty_regularised_lora",
        "base_model_id": MODEL_ID,
        "adapter": "lora",
        "beta": beta,
        "loss": "official quantile loss + n_quantiles * beta * penalty",
        "n_quantiles": N_QUANTILES,
        "coverage_targets": {f"Q{lo:.2f}-Q{hi:.2f}": t
                             for (lo, hi), t in COVERAGE_TARGETS.items()},
        "coverage_smoothing": COVERAGE_SMOOTHING,
        "penalty": "mean over 20 horizons of summed squared positive shortfall",
        "lora_rank": LORA_RANK,
        "lora_alpha": LORA_ALPHA,
        "lora_target_modules": LORA_TARGET_MODULES or "library default",
        "learning_rate": LEARNING_RATE,
        "max_steps": MAX_STEPS,
        "eval_every": EVAL_EVERY,
        "random_seed": RANDOM_SEED,
        "context_length": CONTEXT_LENGTH,
        "min_past": MIN_PAST,
        "prediction_length": PREDICTION_LENGTH,
        "training_dataset": "Dataset E, extended EODHD adjusted closes, 100 M6 assets",
        "training_rows": int(train_array.shape[2]),
        "training_range": f"{DATE_INDEX[0].date()} .. {DATE_INDEX[train_end_pos].date()}",
        "train_target_ceiling": TRAIN_TARGET_CEILING,
        "validation_origin": VALIDATION_ORIGIN,
        "validation_target": f"{VALIDATION_TARGET_START} .. {VALIDATION_TARGET_END}",
        "internal_validation_parameter": VALIDATION_PARAM,
        "trainable_parameters": int(trainable),
        "total_parameters": int(total),
        "trainable_share": round(share, 6),
        "wrapper_calls": int(wrapper_calls),
        "seed_reset_sources": seeded,
        "seed_reset_before_model_load": True,
        "first_call_diagnostics": wrapper.__call_state__["first_call"],
        "training_minutes": round(elapsed, 1),
        "gpu": GPU_NAME,
        "chronos_forecasting": version("chronos-forecasting"),
        "peft": peft.__version__,
        "saved_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
    }

    # --- reload and report validation coverage -------------------------------
    del tuned, fit_result, pipeline
    gc.collect()
    torch.cuda.empty_cache()

    reloaded = Chronos2Pipeline.from_pretrained(
        str(directory), device_map="cuda",
        import_allowlist=["chronos.chronos2.model"],
    )
    assert np.allclose(np.asarray(reloaded.quantiles, dtype=np.float64),
                       QUANTILE_LEVELS), "reloaded quantile levels changed"
    coverage = validation_coverage(reloaded)
    metadata["validation_coverage"] = coverage

    (directory / "training_metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8")
    RUN_SUMMARIES[beta] = metadata

    print(f"  saved and reloaded from {directory}")
    print(f"  validation window, {coverage['valid_observations']} observations:")
    for (lo, hi) in COVERAGE_TARGETS:
        key = f"Q{lo:.2f}-Q{hi:.2f}"
        block = coverage[key]
        print(f"     {key}  nominal {block['nominal']:.0%}  "
              f"target {block['target']:.0%}  "
              f"empirical {block['empirical_coverage']:.4f}  "
              f"mean width {block['mean_width']:.5f}")
    if coverage["quantile_crossings"]:
        print(f"     {'!' * 60}")
        print(f"     WARNING: {coverage['quantile_crossings']} quantile crossings "
              f"in beta {beta}'s validation forecasts.")
        print("     They are left exactly as produced, never sorted or clipped.")
        print("     Review them before interpreting this beta's M6 results.")
        print(f"     {'!' * 60}")
    else:
        print("     no quantile crossings")

    del reloaded
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{len(RUN_SUMMARIES)}/{len(BETAS)} configurations complete.")

## 13. Summary

In [ ]:
print("Chronos-2 uncertainty-regularised LoRA")
print(f"  base model : {MODEL_ID}, reloaded fresh for every beta")
print(f"  adapter    : LoRA rank {LORA_RANK}, alpha {LORA_ALPHA}, "
      f"lr {LEARNING_RATE}, {MAX_STEPS} steps, seed {RANDOM_SEED}")
print(f"  loss       : official quantile loss + {N_QUANTILES} x beta x penalty")
print(f"  targets    : "
      + ", ".join(f"Q{lo:.2f}-Q{hi:.2f} -> {t:.0%}"
                  for (lo, hi), t in COVERAGE_TARGETS.items()))
print(f"  smoothing  : {COVERAGE_SMOOTHING}")
print()
for beta in BETAS:
    meta = RUN_SUMMARIES.get(beta)
    if meta is None:
        print(f"  beta {beta}: NOT COMPLETE")
        continue
    cov = meta.get("validation_coverage", {})
    parts = []
    for (lo, hi) in COVERAGE_TARGETS:
        key = f"Q{lo:.2f}-Q{hi:.2f}"
        if key in cov:
            parts.append(f"{key} {cov[key]['empirical_coverage']:.3f} "
                         f"(w {cov[key]['mean_width']:.4f})")
    print(f"  beta {beta}: {meta['trainable_share']:.2%} trainable, "
          f"{meta.get('training_minutes', 0)} min")
    if parts:
        print(f"           validation {' | '.join(parts)}")
    print(f"           {checkpoint_dir(beta)}")

crossed = [(b, RUN_SUMMARIES[b]["validation_coverage"]["quantile_crossings"])
           for b in BETAS
           if b in RUN_SUMMARIES
           and RUN_SUMMARIES[b].get("validation_coverage", {}).get("quantile_crossings")]
if crossed:
    print(f"\n  {'!' * 66}")
    print("  QUANTILE CROSSINGS present, left uncorrected:")
    for b, n in crossed:
        print(f"    beta {b}: {n} crossings")
    print("  Review these before interpreting the affected beta's M6 results.")
    print(f"  {'!' * 66}")

print("\n  Beta is NOT chosen here, and no winning configuration is declared.")
print("  Validation loss selects the best checkpoint WITHIN each beta, but beta")
print("  changes the objective's scale, so total validation losses are not")
print("  comparable ACROSS 0.5, 1.0 and 2.0. The three are compared later on")
print("  common downstream measurements: M6 RPS, accuracy, confidence, entropy,")
print("  ECE, interval coverage, interval width and quantile crossings.")

print(f"\n  Next: chronos2_uncertainty_lora_m6.ipynb, which runs all three "
      "checkpoints over the 12 M6 origins.")